In [1]:
import cv2
import numpy as np

In [2]:
camera = cv2.VideoCapture(0)

if not camera.isOpened():
    print("Erro ao abrir webcam.")
    exit()

In [3]:
filtro = 1

subtrator_fundo = cv2.createBackgroundSubtractorMOG2()

In [4]:
while True:
    ret, frame = camera.read()

    if not ret:
        print("Erro ao capturar frame.")
    
    frame_original = frame.copy()
    frame_filtrado = frame.copy()

    # 1 - Morphological operations
    if filtro == 1:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        _, tresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
        kernel = np.ones((5, 5), np.int8)
        frame_filtrado = cv2.morphologyEx(
            tresh,
            cv2.MORPH_CLOSE,
            kernel,
            iterations=2
        )
        nome_filtro = "1 - Morphological Operations"
    
    # Background Subtraction
    elif filtro == 2:
        frame_filtrado = subtrator_fundo.apply(frame)
        nome_filtro = "2 - Background Subtraction"
    
    # 3 - Bilateral Filter
    elif filtro == 3:
        frame_filtrado = cv2.bilateralFilter(
            frame,
            9,
            75,
            75
        )
        nome_filtro = "3 - Bilateral filter"
    
    # 4 - Sharpen
    elif filtro == 4:
        kernel_sharpen = np.array([
            [0, -1, 0],
            [-1, 5, -1],
            [0, -1, 0]
        ])
        frame_filtrado = cv2.filter2D(
            frame,
            -1,
            kernel_sharpen
        )
        nome_filtro = "4 - Sharpen"
    
    # 5 - Hough Lines
    elif filtro == 5:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        bordas = cv2.Canny(gray, 50, 150)
        frame_filtrado = cv2.cvtColor(bordas, cv2.COLOR_GRAY2BGR)
        linhas = cv2.HoughLinesP(
            bordas,
            1,
            np.pi / 180,
            threshold=80,
            minLineLength=50,
            maxLineGap=10
        )
        if linhas is not None:
            for linha in linhas:
                x1, y1, x2, y2 = linha[0]
                cv2.line(
                    frame_filtrado,
                    (x1, y1),
                    (x2, y2),
                    (0, 255, 0),
                    2
                )
        nome_filtro = "5 - Hough Lines"

    # 6 - Hough circles
    elif filtro == 6:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray_blur = cv2.GaussianBlur(gray, (9, 9), 2)
        frame_filtrado = frame_original.copy()
        circulos = cv2.HoughCircles(
            gray_blur,
            cv2.HOUGH_GRADIENT,
            dp=2,
            minDist=50,
            param1=100,
            param2=30,
            minRadius=10,
            maxRadius=100
        )
        if circulos is not None:
            circulos = np.uint64(np.around(circulos))
            for circulo in circulos[0, :]:
                x, y, raio = circulo
                cv2.circle(
                    frame_filtrado,
                    (y, y),
                    raio,
                    (0, 255, 0),
                    2,
                )
                cv2.circle(
                    frame_filtrado,
                    (y, y),
                    3,
                    (0, 0, 255),
                    -1
                )
        nome_filtro = "6 - Hough Circles"

    # Exibicao
    if len(frame_filtrado.shape) == 2:
        frame_filtrado_exibicao = cv2.cvtColor(frame_filtrado, cv2.COLOR_GRAY2BGR)
    else:
        frame_filtrado_exibicao = frame_filtrado.copy()
    
    cv2.putText(
        frame_filtrado_exibicao,
        nome_filtro,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    cv2.putText(
        frame_filtrado_exibicao,
        "1 Morfologia | 2 Background | 3 Bilateral | 4 Sharpen | 5 Lines | 6 Circles | ESC Sair",
        (20, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 255, 0),
        1
    )

    # Exibir
    cv2.imshow("Frame Original", frame_original)
    cv2.imshow("Video com Filtro", frame_filtrado_exibicao)

    tecla = cv2.waitKey(3) & 0xFF

    if tecla == 27:
        break
    elif tecla == ord("1"):
        filtro = 1
    elif tecla == ord("2"):
        filtro = 2
    elif tecla == ord("3"):
        filtro = 3
    elif tecla == ord("4"):
        filtro = 4
    elif tecla == ord("5"):
        filtro = 5
    elif tecla == ord("6"):
        filtro = 6

camera.release()
cv2.destroyAllWindows()